# XGBoost -- test

In [ ]:
def evaluate_xgb_baseline(
    #X_aug_train, y_train,
    X_te,  y_test,
    xgb_model: xgb.XGBClassifier,
    label: str = "XGB last-step"
):
    """Evaluate XGBoost on the last time step only (day index 6)."""
    #N_tr, T, D = X_aug_train.shape
    #day_idx_tr = np.full((N_tr, 1), T - 1, dtype=np.float32)  # last step
    #X_tr = np.concatenate([X_aug_train[:, -1, :], day_idx_tr], axis=-1)

    #N_te = X_aug_test.shape[0]
    #day_idx_te = np.full((N_te, 1), T - 1, dtype=np.float32)
    #X_te = np.concatenate([X_aug_test[:, -1, :], day_idx_te], axis=-1)

    probs = xgb_model.predict_proba(X_te)
    preds = probs.argmax(axis=1)

    bacc   = balanced_accuracy_score(y_test, preds)
    y_bin  = label_binarize(y_test, classes=list(range(NUM_CLASSES)))
    auc    = roc_auc_score(y_bin, probs, multi_class="ovr", average="macro")
    f1_macro = f1_score(y_test, preds, average="macro", zero_division=0)
    acc = accuracy_score(y_test, preds)

    #results = np.array([f1_macro, acc, bacc, auc])
    results = np.array([f1_macro, acc])

    report = classification_report(y_test, preds,
                                   target_names=[f"Stage_{i}" for i in range(NUM_CLASSES)],
                                   digits=4)
    log.info(f"\n{'='*60}\n{label} Baseline\n{'='*60}")
    log.info(f"Balanced Accuracy : {bacc:.4f}")
    log.info(f"Macro AUC (OvR)   : {auc:.4f}")
    log.info(f"\n{report}")
    
    
    
# =========================
# Confusion Matrix
# =========================
    cm = confusion_matrix(y_test, preds)

    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")

    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Baseline Confusion Matrix")

    plt.show()

    return results



In [ ]:
xgb_results1 = evaluate_xgb_baseline(X_te_aug[:,-1,:], y_flat_test , xgb_model,  label=f"XGB Baseline " )
